In [ ]:
import pandas as pd
import pandas_ta as ta
import pyotp
from datetime import datetime, timedelta
import time
import csv
import asyncio
import nest_asyncio
import logging
import websocket as ws_client
import xarray as xr

import threading

from NorenRestApiPy.NorenApi import NorenApi

class ShoonyaApiPy(NorenApi):
    def __init__(self):
        NorenApi.__init__(self, host='https://api.shoonya.com/NorenWClientTP/', websocket='wss://api.shoonya.com/NorenWSTP/')        
        global api
        api = self

import logging
#logging.basicConfig(level=logging.DEBUG)
logging.getLogger('websocket').setLevel(logging.INFO)

usercred = pd.read_excel("usercred.xlsx")
user    = usercred.iloc[0,0]
pwd     = usercred.iloc[0,1]
vc      = usercred.iloc[0,2]
app_key = usercred.iloc[0,3]
imei    = usercred.iloc[0,4]
qr = usercred.iloc[0,5]
factor2 = pyotp.TOTP(qr).now()

api = ShoonyaApiPy()
ret = api.login(userid=user, password=pwd, twoFA=factor2, vendor_code=vc, api_secret=app_key, imei=imei)

In [ ]:
import asyncio
import threading
from datetime import datetime
import pandas as pd
import nest_asyncio

feed_opened = False
feedJson = {}
resample_frequency = "1min"
resampling_lock = threading.Lock()
last_resample_time = {}
resampled_data = {}
finalized_data = {}

if 'resampled_data' not in globals():
    resampled_data = {}
if 'last_resample_time' not in globals():
    last_resample_time = {}

def event_handler_feed_update(tick_data):
    try:
        if 'lp' in tick_data and 'tk' in tick_data:
            timest = datetime.fromtimestamp(int(tick_data['ft'])).isoformat()
            token = tick_data['tk']

            with resampling_lock:
                if token not in feedJson:
                    feedJson[token] = []
                    last_resample_time[token] = timest

                feedJson[token].append({'ltp': float(tick_data['lp']), 'tt': timest})
    except (KeyError, ValueError) as e:
        print(f"Error processing tick data: {e}")

async def resample_ticks():
    while True:
        if not feedJson:
            await asyncio.sleep(1)
            continue

        with resampling_lock:
            for token, ticks in feedJson.items():
                try:
                    if ticks:
                        # Create a DataFrame with the new ticks
                        df_new = pd.DataFrame(ticks)
                        df_new["tt"] = pd.to_datetime(df_new["tt"])
                        df_new.set_index("tt", inplace=True)

                        # Determine the current resample interval
                        current_resample_interval = df_new.index.max().floor(resample_frequency)

                        # Initialize or update the resampled DataFrame
                        if token not in resampled_data:
                            resampled_data[token] = pd.DataFrame(index=pd.date_range(end=current_resample_interval, periods=1, freq=resample_frequency), columns=["open", "high", "low", "close"])
                            resampled_data[token].index.name = 'tt'
                            resampled_data[token].loc[current_resample_interval] = [df_new["ltp"].iloc[0], df_new["ltp"].max(), df_new["ltp"].min(), df_new["ltp"].iloc[-1]]
                            last_resample_time[token] = current_resample_interval
                            #print(f"Resampled data for token {token}:\n{resampled_data[token]}")

                        else:
                            # If this interval already exists, update its values
                            if current_resample_interval in resampled_data[token].index:
                                resampled_data[token].loc[current_resample_interval, "high"] = max(
                                    resampled_data[token].loc[current_resample_interval, "high"], df_new["ltp"].max()
                                )
                                resampled_data[token].loc[current_resample_interval, "low"] = min(
                                    resampled_data[token].loc[current_resample_interval, "low"], df_new["ltp"].min()
                                )
                                resampled_data[token].loc[current_resample_interval, "close"] = df_new["ltp"].iloc[-1]
                            else:
                                # If this is a new interval, add a new row for it
                                resampled_data[token].loc[current_resample_interval] = [df_new["ltp"].iloc[0], df_new["ltp"].max(), df_new["ltp"].min(), df_new["ltp"].iloc[-1]]

                            # Print the updated resampled data
                        if current_resample_interval > last_resample_time[token]:  # Check if a new minute has started                           
                            
                            last_resample_time[token] = current_resample_interval
                        
                       
                    feedJson[token] = []

                except Exception as e:
                    if isinstance(e, KeyError) and e.args[0] == 'tt':
                        print(f"Market likely closed for token {token}")
                    else:
                        print(f"Error resampling data for token {token}: {e}")
                finally:    
                    feedJson[token] = []
        await asyncio.sleep(1)
        

def event_handler_order_update(tick_data):
    print(f"Order update {tick_data}")

def open_callback():
    global feed_opened
    feed_opened = True

# Assuming `api` is defined and starts the WebSocket connection
api.start_websocket(
    order_update_callback=event_handler_order_update,
    subscribe_callback=event_handler_feed_update,
    socket_open_callback=open_callback
)

while not feed_opened:
    pass

api.subscribe(['MCX|428009', 'NSE|26009'])

async def main():
    await resample_ticks()

loop = asyncio.get_event_loop()

if loop.is_running():
    nest_asyncio.apply()
    asyncio.create_task(main())
else:
    loop.run_until_complete(main())


In [1]:
resampled_data


NameError: name 'resampled_data' is not defined

In [ ]:
import csv

# Assuming last_resampled_data is defined and populated with data

# Specify the CSV file path
csv_file = 'last_resampled_data1.csv'

try:
    # Open the CSV file in write mode
    with open(csv_file, 'w', newline='') as file:
        writer = csv.writer(file)

        # Write headers for each token's data
        writer.writerow(['Token', 'Timestamp', 'Open', 'High', 'Low', 'Close', 'EMA1', 'EMA2' , 'EMA3'])

        # Iterate through each token and its corresponding DataFrame
        for token, df in last_resampled_data.items():
            # Extract relevant columns for writing to CSV
            for index, row in df.iterrows():
                writer.writerow([token, index, row['open'], row['high'], row['low'], row['close'], row['ema1'], row['ema2'], row['ema3']])

    print(f"Data successfully written to {csv_file}")

except Exception as e:
    print(f"Error writing data to CSV: {e}")


In [ ]:
feedJson